# Analyse de parties — Catan (version simplifiée)

Ce notebook charge un **ensemble de parties** sauvegardées au format **JSONL** par le
moteur (`play.py` / `moteur.py`) et en produit de nombreuses **visualisations Plotly**
détaillées : comportement des joueurs, économie du marché, ressources, points de
victoire, constructions, voleur, etc.

Il est **réutilisable sur n'importe quel jeu de parties** : il suffit de pointer
`DOSSIER` vers le dossier contenant les fichiers `*.jsonl`.

**Format attendu** (une ligne JSON par enregistrement) :
- `meta` : noms/types des joueurs et **plateau statique** (tuiles, sommets, arêtes) ;
- `step` : à chaque décision, l'`observation` du joueur actif, les `actions_legales`
  et l'`action` choisie ;
- `resultat` : gagnant et points finaux.

> Astuce : exécutez les cellules dans l'ordre (*Run All*). Ce notebook produit des
> **statistiques agrégées** sur l'ensemble des parties ; pour analyser une **partie
> précise** (plateau, marché, courbes), utilisez le visualiseur `python view.py`.


## 0. Configuration et imports

In [1]:
import os, glob, json, math, pickle, hashlib, time
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# ----------------------------- À PERSONNALISER ------------------------------
DOSSIER = "sauvegardes"      # dossier contenant les fichiers *.jsonl
UTILISER_CACHE = True        # met en cache les tables (rechargement quasi instantané)

# Génération automatique si le dossier est vide (utilise le moteur local).
GENERER_SI_VIDE = True
GEN_N_PARTIES   = 40
GEN_TYPES       = ["rl", "random", "random", "random"]
GEN_SEED        = 1
# ----------------------------------------------------------------------------

# JSON rapide (orjson) si disponible — sinon repli sur la bibliothèque standard.
try:
    import orjson
    def _loads(ligne): return orjson.loads(ligne)
    JSON_BACKEND = "orjson"
except ImportError:
    def _loads(ligne): return json.loads(ligne)
    JSON_BACKEND = "json (installez 'orjson' pour ~3x plus rapide)"

pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Set2
pd.set_option("display.max_columns", 50)
print("Backend JSON :", JSON_BACKEND)


Backend JSON : orjson


## 1. (Optionnel) Génération de parties

Si `DOSSIER` ne contient aucun fichier `*.jsonl` et que `GENERER_SI_VIDE` est `True`,
on génère un échantillon avec le moteur local (`moteur.py`, `joueur.py`).

In [2]:
def generer_parties(dossier, n_parties, types, seed=1, cycle=True):
    from joueur import creer_joueur
    from moteur import Moteur
    os.makedirs(dossier, exist_ok=True)
    for partie in range(n_parties):
        dec = (partie % len(types)) if cycle else 0
        tp = types[dec:] + types[:dec]
        sp = seed + partie
        joueurs = [creer_joueur(t, nom=f"{t}#{i}", seed=sp*100+i)
                   for i, t in enumerate(tp)]
        chemin = os.path.join(dossier, f"partie_{partie:05d}.jsonl")
        Moteur(joueurs, seed=sp, chemin_sauvegarde=chemin,
               partie_id=partie).jouer_partie()

if not glob.glob(os.path.join(DOSSIER, "*.jsonl")) and GENERER_SI_VIDE:
    print(f"Aucune partie dans {DOSSIER!r} : génération de {GEN_N_PARTIES} parties…")
    generer_parties(DOSSIER, GEN_N_PARTIES, GEN_TYPES, GEN_SEED)
    print("Génération terminée.")
else:
    print("Parties déjà présentes (ou génération désactivée).")


Parties déjà présentes (ou génération désactivée).


## 2. Chargement et mise en tables

On lit tous les fichiers et on construit plusieurs `DataFrame` :
`df_parties` (une ligne par partie), `df_steps` (une ligne par décision),
`df_series` (une ligne par joueur et par instant, pour les séries temporelles),
`df_marche` (prix du marché), `df_robber` (déplacements du voleur).

**Performance.** Le chargement est optimisé : lecture binaire + parseur rapide
(`orjson` si présent), construction *colonne par colonne* (bien plus rapide que
ligne par ligne), et **cache** sur disque (`_cache_tables.pkl`). Le premier
chargement construit le cache ; les suivants sont quasi instantanés tant que les
fichiers `.jsonl` ne changent pas. Mettez `UTILISER_CACHE = False` pour forcer la
reconstruction.

In [3]:
def _extraire_fichier(path):
    """Parse un fichier (une partie) et renvoie ses données en colonnes."""
    meta = resultat = board = None
    final = {}
    types = []
    S = {k: [] for k in ("partie", "t", "phase", "joueur", "type_joueur", "de",
                         "action_type", "action_res", "action_sommet", "or",
                         "cartes_pv", "cartes_chevalier", "plus_grande_armee",
                         "route_la_plus_longue")}
    Se = {k: [] for k in ("partie", "t", "joueur", "type_joueur", "pv_public",
                          "ressources_total", "chevaliers_joues", "longueur_route")}
    M = {k: [] for k in ("partie", "t", "res", "prix")}
    R = {k: [] for k in ("partie", "t", "q", "r")}
    with open(path, "rb") as fh:                       # binaire = plus rapide
        for ligne in fh:
            if not ligne.strip():
                continue
            o = _loads(ligne)
            ty = o["type"]
            if ty == "meta":
                meta = o; types = o.get("types_joueurs", [])
                board = o.get("plateau_statique", {}); continue
            if ty == "resultat":
                resultat = o; continue
            pid = o["partie"]; n = len(types); tt = o["t"]
            obs = o.get("observation", {}); act = o.get("action", {})
            atype = act.get("type"); jact = o.get("joueur")
            final = {"graphe": obs.get("graphe", {}),
                     "plus_grande_armee": obs.get("plus_grande_armee", -1),
                     "route_la_plus_longue": obs.get("route_la_plus_longue", -1)}
            S["partie"].append(pid); S["t"].append(tt); S["phase"].append(o.get("phase"))
            S["joueur"].append(jact); S["type_joueur"].append(types[jact] if types else None)
            S["de"].append(o.get("de")); S["action_type"].append(atype)
            S["action_res"].append(act.get("res") or act.get("donne"))
            S["action_sommet"].append(act.get("sommet"))
            S["or"].append(obs.get("or")); S["cartes_pv"].append(obs.get("cartes_pv"))
            S["cartes_chevalier"].append(obs.get("cartes_chevalier"))
            S["plus_grande_armee"].append(obs.get("plus_grande_armee"))
            S["route_la_plus_longue"].append(obs.get("route_la_plus_longue"))
            pv = obs.get("pv_publics") or [None]*n
            ra = obs.get("ressources_adversaires") or [None]*n
            cj = obs.get("chevaliers_joues") or [None]*n
            lr = obs.get("longueur_routes") or [None]*n
            for jdx in range(n):
                Se["partie"].append(pid); Se["t"].append(tt); Se["joueur"].append(jdx)
                Se["type_joueur"].append(types[jdx]); Se["pv_public"].append(pv[jdx])
                Se["ressources_total"].append(ra[jdx]); Se["chevaliers_joues"].append(cj[jdx])
                Se["longueur_route"].append(lr[jdx])
            for r, prix in (obs.get("prix_marche") or {}).items():
                M["partie"].append(pid); M["t"].append(tt); M["res"].append(r); M["prix"].append(prix)
            if atype == "voleur":
                q, r = act["tuile"]
                R["partie"].append(pid); R["t"].append(tt); R["q"].append(q); R["r"].append(r)
    pid = meta["partie"] if meta else None
    return {"pid": pid, "meta": meta, "resultat": resultat, "board": board,
            "final": final, "S": S, "Se": Se, "M": M, "R": R}


def _construire_depuis_fichiers(fichiers):
    metas, resultats, boards, final_obs = {}, {}, {}, {}
    extraits = [_extraire_fichier(f) for f in fichiers]
    cols = {nom: {k: [] for k in extraits[0][nom]} for nom in ("S", "Se", "M", "R")}
    p_rows = []
    for d in extraits:
        pid = d["pid"]
        metas[pid] = d["meta"]; resultats[pid] = d["resultat"]
        boards[pid] = d["board"]; final_obs[pid] = d["final"]
        for nom in ("S", "Se", "M", "R"):
            for k in cols[nom]:
                cols[nom][k].extend(d[nom][k])
        meta = d["meta"] or {}; res = d["resultat"] or {}
        types = meta.get("types_joueurs", []); g = res.get("gagnant")
        p_rows.append({"partie": pid, "n_joueurs": len(types), "types": types,
                       "noms": meta.get("joueurs", []), "gagnant": g,
                       "type_gagnant": types[g] if g is not None and types else None,
                       "points": res.get("points")})

    df_steps   = pd.DataFrame(cols["S"])
    df_series  = pd.DataFrame(cols["Se"])
    df_marche  = pd.DataFrame(cols["M"])
    df_robber  = pd.DataFrame(cols["R"])
    df_parties = pd.DataFrame(p_rows).sort_values("partie").reset_index(drop=True)

    # Détection des tours (hors préparation) : un tour = run de steps d'un même joueur.
    npx = df_steps[df_steps["phase"] != "PREP"].copy()
    npx["nt"] = npx["joueur"] != npx.groupby("partie")["joueur"].shift()
    npx["tour_id"] = npx.groupby("partie")["nt"].cumsum()
    df_steps = df_steps.merge(npx[["partie", "t", "nt", "tour_id"]],
                              on=["partie", "t"], how="left")
    nb_steps = df_steps.groupby("partie").size().rename("nb_steps")
    nb_tours = npx.groupby("partie")["tour_id"].max().rename("nb_tours")
    df_parties = (df_parties.merge(nb_steps, on="partie", how="left")
                            .merge(nb_tours, on="partie", how="left"))

    # Ressource + numéro de la tuile ciblée par le voleur (vectorisé).
    if not df_robber.empty:
        res_t, num_t = {}, {}
        for pid, b in boards.items():
            for tt in (b or {}).get("tuiles", []):
                res_t[(pid, tt["q"], tt["r"])] = tt["res"]
                num_t[(pid, tt["q"], tt["r"])] = tt["num"]
        cles = list(zip(df_robber["partie"], df_robber["q"], df_robber["r"]))
        df_robber["res"] = [res_t.get(k) for k in cles]
        df_robber["num"] = [num_t.get(k) for k in cles]

    return {"df_parties": df_parties, "df_steps": df_steps, "df_series": df_series,
            "df_marche": df_marche, "df_robber": df_robber, "boards": boards,
            "final_obs": final_obs, "npx": npx}


_CACHE_VERSION = 1
def _signature(fichiers):
    items = [(os.path.basename(f), os.path.getsize(f), int(os.path.getmtime(f)))
             for f in fichiers]
    return hashlib.md5((str(_CACHE_VERSION) + repr(items)).encode()).hexdigest()


def charger_tables(dossier, utiliser_cache=True):
    fichiers = sorted(glob.glob(os.path.join(dossier, "*.jsonl")))
    if not fichiers:
        raise FileNotFoundError(f"Aucun fichier .jsonl dans {dossier!r}.")
    cache = os.path.join(dossier, "_cache_tables.pkl")
    sig = _signature(fichiers)
    if utiliser_cache and os.path.exists(cache):
        try:
            with open(cache, "rb") as f:
                blob = pickle.load(f)
            if blob.get("signature") == sig:
                print(f"Tables chargées depuis le cache ({len(fichiers)} fichiers).")
                return blob["T"], fichiers
        except Exception:
            pass
    t0 = time.perf_counter()
    T = _construire_depuis_fichiers(fichiers)
    print(f"Tables construites en {time.perf_counter()-t0:.1f}s "
          f"({len(fichiers)} fichiers, backend {JSON_BACKEND.split()[0]}).")
    if utiliser_cache:
        try:
            with open(cache, "wb") as f:
                pickle.dump({"signature": sig, "T": T}, f, protocol=pickle.HIGHEST_PROTOCOL)
        except Exception as e:
            print("Cache non écrit :", e)
    return T, fichiers


T, fichiers = charger_tables(DOSSIER, UTILISER_CACHE)
df_parties = T["df_parties"]; df_steps = T["df_steps"]; df_series = T["df_series"]
df_marche  = T["df_marche"];  df_robber = T["df_robber"]
boards = T["boards"]; final_obs = T["final_obs"]; npx = T["npx"]

N = int(df_parties["n_joueurs"].max())

print(f"{len(fichiers)} fichiers | {len(df_parties)} parties | {len(df_steps)} décisions "
      f"| {N} joueurs max")
df_parties[["partie", "type_gagnant", "nb_steps", "nb_tours", "points"]].head()


Tables construites en 2.9s (100 fichiers, backend orjson).
100 fichiers | 100 parties | 59125 décisions | 4 joueurs max


,partie,type_gagnant,nb_steps,nb_tours,points
0,0,Simple,298,72,"[3, 6, 4, 10]"
1,1,Simple2,651,174,"[3, 10, 4, 9]"
2,2,Simple,497,160,"[9, 3, 4, 10]"
3,3,Simple,400,77,"[10, 4, 8, 8]"
4,4,Simple2,362,107,"[5, 2, 10, 5]"


## 3. Vue d'ensemble

Qui gagne, avec combien de points, et en combien de temps.

In [4]:
vt = df_parties["type_gagnant"].value_counts().reset_index()
vt.columns = ["type", "victoires"]
fig = px.bar(vt, x="type", y="victoires", color="type", text="victoires",
             title="Victoires par type de joueur", color_discrete_sequence=PALETTE)
fig.show()

vp = df_parties["gagnant"].value_counts().sort_index().reset_index()
vp.columns = ["position", "victoires"]
px.bar(vp, x="position", y="victoires", text="victoires",
       title="Victoires par position de départ (siège)").show()

In [5]:
# Points finaux par type de joueur
pts_rows = []
for _, row in df_parties.iterrows():
    if isinstance(row["points"], list):
        for jdx, pt in enumerate(row["points"]):
            pts_rows.append({"partie": row["partie"], "joueur": jdx,
                             "type_joueur": row["types"][jdx], "points": pt})
df_pts = pd.DataFrame(pts_rows)
px.box(df_pts, x="type_joueur", y="points", color="type_joueur", points="all",
       title="Distribution des points finaux par type de joueur",
       color_discrete_sequence=PALETTE).show()

In [6]:
# Durée des parties
px.histogram(df_parties, x="nb_tours", nbins=30,
             title="Durée des parties (nombre de tours)").show()
px.histogram(df_parties, x="nb_steps", nbins=30,
             title="Durée des parties (nombre de décisions)").show()

## 4. Lancers de dés

On reconstitue **un lancer par tour** (les `step` d'un même tour partagent la même
valeur `de`) et on compare à la loi théorique de 2d6.

In [7]:
des = npx[npx["nt"]]["de"].dropna().astype(int)
dd = des.value_counts(normalize=True).sort_index().reset_index()
dd.columns = ["de", "freq"]
theo = {2:1,3:2,4:3,5:4,6:5,7:6,8:5,9:4,10:3,11:2,12:1}
dd["theorique"] = dd["de"].map(lambda x: theo.get(x, 0)/36)

fig = go.Figure()
fig.add_bar(x=dd["de"], y=dd["freq"], name="observé", marker_color="#4C78A8")
fig.add_scatter(x=dd["de"], y=dd["theorique"], name="théorique 2d6",
                mode="lines+markers", line=dict(color="crimson"))
fig.update_layout(title="Distribution des lancers de dés — observé vs théorique",
                  xaxis_title="somme des dés", yaxis_title="fréquence")
fig.show()

## 5. Comportement des joueurs (actions)

Quelles actions sont jouées, et comment le *mix* d'actions diffère selon le type de
joueur.

In [8]:
af = df_steps["action_type"].value_counts().reset_index()
af.columns = ["action", "n"]
px.bar(af, x="action", y="n", title="Fréquence des types d'action (toutes parties)").show()

mix = df_steps.groupby(["type_joueur", "action_type"]).size().reset_index(name="n")
mix["prop"] = mix.groupby("type_joueur")["n"].transform(lambda s: s/s.sum())
px.bar(mix, x="type_joueur", y="prop", color="action_type", barmode="stack",
       title="Répartition (proportion) des actions par type de joueur").show()

In [9]:
# Heatmap action x type de joueur
ht = df_steps.groupby(["type_joueur", "action_type"]).size().reset_index(name="n")
ht["prop"] = ht.groupby("type_joueur")["n"].transform(lambda s: s/s.sum())
pivot = ht.pivot(index="action_type", columns="type_joueur", values="prop").fillna(0)
px.imshow(pivot, text_auto=".2f", aspect="auto", color_continuous_scale="Blues",
          title="Proportion de chaque action selon le type de joueur").show()

# Décisions par tour
apt = npx.groupby(["partie", "tour_id"]).size().reset_index(name="n_actions")
px.histogram(apt, x="n_actions", nbins=30,
             title="Nombre de décisions par tour").show()

## 6. Marché central

Prix finaux moyens, volumes et flux net achats/ventes, **agrégés sur toutes les
parties**. (L'évolution des prix d'une partie précise est dans le visualiseur
`view.py`.)

In [10]:
prix_final = df_marche.sort_values("t").groupby(["partie", "res"]).tail(1)
pfm = prix_final.groupby("res")["prix"].mean().reset_index()
px.bar(pfm, x="res", y="prix", color="res",
       title="Prix final moyen par ressource (toutes parties)").show()

In [11]:
vol = df_steps[df_steps["action_type"].isin(["marche_acheter", "marche_vendre"])]
vg = vol.groupby(["action_type", "action_res"]).size().reset_index(name="n")
px.bar(vg, x="action_res", y="n", color="action_type", barmode="group",
       title="Volume d'achats / ventes par ressource (marché)").show()

flux = vol.copy()
flux["signe"] = flux["action_type"].map({"marche_acheter": 1, "marche_vendre": -1})
fn = flux.groupby("action_res")["signe"].sum().reset_index(name="flux_net")
px.bar(fn, x="action_res", y="flux_net", color="action_res",
       title="Flux net du marché par ressource (achats − ventes)").show()

## 7. Trajectoires d'une partie → visualiseur

Les **séries temporelles d'une partie individuelle** (or, points de victoire,
route la plus longue, chevaliers joués, ressources, prix) se consultent désormais
dans le **visualiseur** interactif, onglets « Courbes d'évolution », avec une bascule
*par étape / par tour* :

```bash
python view.py            # puis « Charger une sauvegarde »
```

Ce notebook se concentre sur les **statistiques agrégées** sur l'ensemble des parties.

## 8. Constructions et développement

Routes, colonies, villes, cartes développement et chevaliers joués.

In [12]:
builds = df_steps[df_steps["action_type"].isin(
    ["construire_route", "construire_colonie", "construire_ville",
     "acheter_dev", "jouer_chevalier"])]
bc = builds.groupby(["type_joueur", "action_type"]).size().reset_index(name="n")
px.bar(bc, x="action_type", y="n", color="type_joueur", barmode="group",
       title="Constructions / achats par type de joueur",
       color_discrete_sequence=PALETTE).show()

## 9. Voleur

Où va le voleur (agrégé sur toutes les parties) ? Par ressource ciblée et par numéro
de tuile. (La carte du voleur d'une partie précise est dans le visualiseur.)

In [13]:
if not df_robber.empty:
    rr = df_robber["res"].value_counts().reset_index()
    rr.columns = ["res", "n"]
    px.bar(rr, x="res", y="n", color="res",
           title="Placements du voleur par ressource ciblée").show()

    rn = df_robber.dropna(subset=["num"])["num"].astype(int).value_counts().sort_index().reset_index()
    rn.columns = ["num", "n"]
    px.bar(rn, x="num", y="n", title="Cibles du voleur par numéro de tuile").show()
else:
    print("Aucun déplacement de voleur enregistré.")

## 10. Placements initiaux

Quelles ressources les joueurs recherchent-ils pour leurs **colonies de départ** ?

In [14]:
prep = df_steps[df_steps["action_type"] == "prep_colonie"].dropna(subset=["action_sommet"])
pref_rows = []
for _, row in prep.iterrows():
    b = boards.get(row["partie"], {})
    sid = int(row["action_sommet"])
    sommets = b.get("sommets", [])
    if sid < len(sommets):
        for couple in sommets[sid].get("tuiles", []):
            pref_rows.append({"res": couple[0], "num": couple[1]})
if pref_rows:
    dpf = pd.DataFrame(pref_rows)
    px.bar(dpf["res"].value_counts().reset_index().set_axis(["res", "n"], axis=1),
           x="res", y="n", color="res",
           title="Ressources adjacentes aux colonies initiales").show()
    px.histogram(dpf.dropna(subset=["num"]), x="num",
                 title="Numéros de tuile recherchés au placement initial").show()

## 11. Corrélations

Relations entre l'état final d'un joueur (constructions, chevaliers) et ses points.

In [15]:
final = df_series.sort_values("t").groupby(["partie", "joueur"]).tail(1)
agg_builds = (builds.groupby(["partie", "joueur", "action_type"]).size()
              .unstack(fill_value=0).reset_index())
corr = final.merge(agg_builds, on=["partie", "joueur"], how="left").fillna(0)
corr = corr.merge(df_pts[["partie", "joueur", "points"]], on=["partie", "joueur"], how="left")
for c in ["construire_route", "construire_colonie", "construire_ville", "acheter_dev"]:
    if c not in corr: corr[c] = 0

px.scatter(corr, x="construire_colonie", y="points", color="type_joueur",
           title="Points finaux vs colonies construites",
           color_discrete_sequence=PALETTE).show()
px.scatter(corr, x="chevaliers_joues", y="points", color="type_joueur",
           title="Points finaux vs chevaliers joués",
           color_discrete_sequence=PALETTE).show()
px.scatter(corr, x="longueur_route", y="points", color="type_joueur",
           title="Points finaux vs longueur de route",
           color_discrete_sequence=PALETTE).show()

## 12. Synthèse chiffrée

Tableau récapitulatif par type de joueur.

In [16]:
synth = (df_pts.groupby("type_joueur")
         .agg(parties=("points", "size"),
              points_moyen=("points", "mean"),
              points_med=("points", "median"),
              points_max=("points", "max"))
         .reset_index())
wins = df_parties["type_gagnant"].value_counts().rename("victoires")
synth = synth.merge(wins, left_on="type_joueur", right_index=True, how="left").fillna(0)
synth["taux_victoire"] = (synth["victoires"] / synth["parties"]).round(3)
synth.round(2)

,type_joueur,parties,points_moyen,points_med,points_max,victoires,taux_victoire
0,JoueurRL,100,4.00,4.0,10,1,0.01
1,Simple,200,6.71,6.0,11,62,0.31
2,Simple2,100,7.69,8.0,11,37,0.37


---
*Notebook réutilisable : changez `DOSSIER` pour analyser un autre ensemble de parties.
Pour inspecter une partie précise, utilisez le visualiseur `python view.py`.*